In [ ]:
# Inference on a single XRD file to predict max magnetization of iron oxide nanoparticles
import numpy as np
import pandas as pd
from pathlib import Path

# User inputs

# XRD file 
XRD_DIR  = r"C:\Users\fabel\Documents\materials-project\notesbooks\Iron_Oxide_Model\experimanta_data"       
XRD_FILE = r"2007_XRD.csv"        

# Model file 
MODEL_DIR  = r"C:\Users\fabel\Documents\materials-project\notesbooks\Iron_Oxide_Model"  
MODEL_FILE = r"rf_model_fe3o4_feo_dataset_N_1000_v2.pkl"            

PRED_ERROR_VALUE = 4.5        # RMSE or MAE from prediction on heldout simulated data
PRED_ERROR_NAME  = "RMSE"     

# Preprocessing conditions
CLIP_NONNEG  = True            # set negatives to zero
CROP_TO_GRID = True            # crop XRD to training grid range before resampling

# Load CSV file, XRD CSV with either named columns (2theta/intensity) or first two columns
def smart_read_xrd_csv(path):
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    th_keys = [k for k in cols if k in ["2theta", "two_theta", "angle", "2θ", "theta"]]
    I_keys  = [k for k in cols if k in ["intensity", "counts", "i", "y"]]
    if th_keys and I_keys:
        x = df[cols[th_keys[0]]].to_numpy(dtype=float)
        y = df[cols[I_keys[0]]].to_numpy(dtype=float)
    else:
        x = df.iloc[:, 0].to_numpy(dtype=float)
        y = df.iloc[:, 1].to_numpy(dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    order = np.argsort(x)
    return x[order], y[order]

def resample_to_grid(x, y, grid, fill_value=0.0):
    y_grid = np.interp(grid, x, y, left=np.nan, right=np.nan)
    y_grid[grid < x.min()] = fill_value
    y_grid[grid > x.max()] = fill_value
    return np.nan_to_num(y_grid, nan=fill_value, posinf=fill_value, neginf=fill_value)

def normalize_max(y, eps=1e-12):
    y = np.asarray(y, float)
    m = np.nanmax(y)
    return y / (m if m > eps else eps)

# Model lorder
def load_model(path):
    import joblib, gzip, bz2, lzma, pickle as _p
    p = Path(path)
    try:
        return joblib.load(p)
    except Exception:
        pass
    try:
        with open(p, "rb") as f:
            return _p.load(f)
    except Exception:
        pass
    for opener in (gzip.open, bz2.open, lzma.open):
        try:
            with opener(p, "rb") as f:
                return _p.load(f)
        except Exception:
            continue
    with open(p, "rb") as f:
        head = f.read(8)
    raise RuntimeError(f"Could not load model {p}. Header: {head!r}")

#Preprocessing:
# True/False, crop to training grid range
# True/False, clip negatives
# Normalize to max = 1
# Resample to the training grid
# Normalize again to max = 1 (post-interp normalization)
def process_xrd_for_prediction_simple(x_path, grid, *,
                                      clip_nonneg=CLIP_NONNEG,
                                      crop_to_grid=CROP_TO_GRID):
    x_raw, y_raw = smart_read_xrd_csv(x_path)
    grid_min, grid_max = float(np.min(grid)), float(np.max(grid))

    if crop_to_grid:
        m = (x_raw >= grid_min) & (x_raw <= grid_max)
        x_use, y_use = x_raw[m], y_raw[m]
        if x_use.size < 7:
            raise ValueError(f"{x_path}: too few points in training grid range.")
    else:
        x_use, y_use = x_raw, y_raw

    if clip_nonneg:
        y_use = np.clip(y_use, 0, None)

    y_norm = normalize_max(y_use)
    y_grid = resample_to_grid(x_use, y_norm, grid, fill_value=0.0)
    y_grid = normalize_max(y_grid)
    return y_grid

# Main prediction function
def main():
    xrd_path   = Path(XRD_DIR)   / XRD_FILE
    model_path = Path(MODEL_DIR) / MODEL_FILE

    if not xrd_path.exists():
        raise FileNotFoundError(f"Missing XRD file: {xrd_path}")
    if not model_path.exists():
        raise FileNotFoundError(f"Missing model file: {model_path}")

    two_theta_grid = np.linspace(25.0, 65.0, 2000)

    model = load_model(model_path)

    n_in = getattr(model, "n_features_in_", None)
    if n_in is not None and int(n_in) != two_theta_grid.size:
        raise RuntimeError(
            f"Model expects n_features_in_={n_in}, but grid has {two_theta_grid.size} points. "
            "Confirm the fixed grid matches what the model was trained on."
        )

    # XRD t0 feature vector
    y_grid = process_xrd_for_prediction_simple(xrd_path, two_theta_grid)

    # Predict
    X_feat = y_grid.reshape(1, -1)   # shape (1, n_features)
    pred   = float(np.ravel(model.predict(X_feat))[0])

    # Print result with provided uncertainty
    print("==============================================")
    print("XRD inference")
    print("----------------------------------------------")
    print(f"XRD file : {xrd_path.name}  (dir: {xrd_path.parent})")
    print(f"Model    : {model_path.name}  (dir: {model_path.parent})")
    print("----------------------------------------------")
    print(f"Predicted max magnetization: {pred:.3g} Am^2/kg")
    print(f"Reported uncertainty      : ±{PRED_ERROR_VALUE:.1g} Am^2/kg ({PRED_ERROR_NAME})")
    print("==============================================")

if __name__ == "__main__":
    main()